In [ ]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
from pathlib import Path

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root so repo-root-relative CONFIG_PATH values resolve
# regardless of how the notebook was launched (jupyter CWD = notebook dir,
# scheduler = repo root). The lab ALWAYS lives at
# ``<repo_root>/research_notebooks/bowaka_v2_lab``, so derive the repo root from
# that canonical layout. A marker-only heuristic ("a dir with research_notebooks/
# AND Makefile") mis-matches the lab dir itself when it carries a Makefile and a
# stray nested research_notebooks/ — which then chdir's one level too deep and
# breaks every repo-root-relative path.
if _lab_root.parent.name == "research_notebooks":
    _repo_root = _lab_root.parent.parent
else:
    # Fallback: the repo root holds research_notebooks/bowaka_common (the sibling
    # package) — a marker a stray nested research_notebooks/ inside the lab lacks.
    _repo_root = _lab_root
    for _candidate in [_lab_root, *_lab_root.parents]:
        if (_candidate / "research_notebooks" / "bowaka_common").is_dir():
            _repo_root = _candidate
            break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


In [ ]:
# Papermill parameters — override via `papermill -p <name> <value>` or in-place.
START_DATE = '2026-05-19'      # parity-window inclusive start (ISO date)
END_DATE   = '2026-05-23'      # parity-window inclusive end   (ISO date)
SYMBOLS    = None              # optional list[str]; default = small PIT sample
PROD_CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/reference/source_strategy/scripts/bowaka_v2_config.yaml'
LAB_CONFIG_PATH  = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_actual_iex_current_code.yml'
LAKE_ROOT  = None              # default = bowaka_common.resolve_market_data_root()
COST_STRESS = 'conservative'   # passed to both sides identically
RUN_LABEL  = None              # default = UTC timestamp folder name


# 13 — Production-vs-Lab Parity

Empirical agreement between the production-side backtester
(`reference/source_strategy/scripts/bowaka_v2_backtest.py`) and the lab's
`run_config_backtest` over a chosen window. Audit §14.5 thresholds drive the
stop-ship verdict; failing rows point at a specific divergence class.

**Requires Phase 0's fix landed** — pre-fix the production side always read
deterministic synthetic data and the parity metrics are meaningless.

In [ ]:
# Resolve lake root + a sensible default universe.
import datetime as _dt
from pathlib import Path

from bowaka_common.marketdata.store import resolve_market_data_root
from bowaka_common.marketdata.catalog import available_symbols

_lake_root = Path(LAKE_ROOT).resolve() if LAKE_ROOT else resolve_market_data_root(None, create=False)
print(f'lake_root: {_lake_root}')

_start = _dt.date.fromisoformat(START_DATE)
_end   = _dt.date.fromisoformat(END_DATE)

if SYMBOLS is None:
    # Default: first 5 IEX split_adjusted symbols on disk — small, fast, real.
    _syms = available_symbols(_lake_root, timeframe='1d', vendor='alpaca',
                              feed='iex', adjustment='split_adjusted')[:5]
else:
    _syms = [str(s) for s in SYMBOLS]
print(f'universe: {_syms} ({len(_syms)} symbols)')
print(f'window:   {_start} -> {_end}')


In [ ]:
# Run both sides + compute parity.
from bowaka_v2_lab.parity import run_parity, render_markdown_report

_label = RUN_LABEL or _dt.datetime.now(_dt.UTC).strftime('%Y%m%dT%H%M%SZ')
_run_root = Path('research_notebooks/bowaka_v2_lab/artifacts/parity/lab_vs_production') / _label
_run_root.mkdir(parents=True, exist_ok=True)
print(f'run_root: {_run_root}')

report = run_parity(
    start_date=_start, end_date=_end,
    symbols=_syms,
    prod_config_path=Path(PROD_CONFIG_PATH),
    lab_config_path=Path(LAB_CONFIG_PATH),
    lake_root=_lake_root,
    cost_stress=COST_STRESS,
    run_root=_run_root,
)
print(f'prod_n_trades={report.prod_n_trades}  lab_n_trades={report.lab_n_trades}')
print(f'trade_intersection_rate={report.trade_intersection_rate:.4f}')
print(f'fill_price_mae_bps={report.fill_price_mae_bps:.4f}')
print(f'passes_audit_thresholds={report.passes_audit_thresholds}')
if report.failing_metrics:
    print(f'failing metrics: {report.failing_metrics}')


In [ ]:
# Persist the paste-back Markdown.
_md_path = _run_root / 'parity_report.md'
render_markdown_report(report, output_path=_md_path)
print(f'wrote: {_md_path}')
print()
print(_md_path.read_text(encoding='utf-8'))
